# Linux Process Management

---

## The Core Concept (Lock This In First)

A **process** is a running instance of a program. Every process has:
- A **PID** (Process ID) — unique integer assigned by the kernel
- A **PPID** (Parent Process ID) — who spawned it
- An **owner** (UID) — who it runs as
- A **state** — running, sleeping, stopped, zombie
- **File descriptors** — open files, sockets, pipes
- **CPU and memory allocation**

The kernel manages all processes. You interact with them via **signals** and **process management tools**.

---

## Process States

| State | Symbol | Meaning |
|---|---|---|
| Running | `R` | Actively using CPU or ready to run |
| Sleeping (interruptible) | `S` | Waiting for event (I/O, timer) — can be woken by signal |
| Sleeping (uninterruptible) | `D` | Waiting for I/O — cannot be killed (disk/NFS wait) |
| Stopped | `T` | Paused by signal (SIGSTOP/Ctrl+Z) |
| Zombie | `Z` | Finished but parent hasn't read its exit code yet |

> **Zombie processes** can't be killed directly — you kill the parent. If the parent is dead, init (PID 1) adopts and reaps them. A system full of zombies indicates a parent process not calling `wait()`.

> **Uninterruptible sleep (D)** is the one state where `SIGKILL` doesn't work — process is stuck in kernel space waiting for I/O. Usually resolves when the I/O completes or times out.

---

## `ps` — Process Status (Snapshot)

`ps` gives a **point-in-time snapshot** of running processes.

```bash
# ─── Most Used Commands ────────────────────────────────────────
ps aux                    # ALL processes, ALL users, full details (BSD syntax)
ps -ef                    # ALL processes, full format (UNIX syntax) — same result
ps aux | grep nginx       # find a specific process
ps aux | grep -v grep     # exclude the grep process itself from results

# ─── Tree View (shows parent-child relationships) ──────────────
ps auxf                   # forest/tree format with ASCII art
ps -ejH                   # process hierarchy, UNIX style
pstree                    # dedicated tree viewer (cleaner output)
pstree -p                 # tree with PIDs shown

# ─── Sort by resource usage ────────────────────────────────────
ps aux --sort=-%cpu       # sort by CPU usage, highest first
ps aux --sort=-%mem       # sort by memory usage, highest first
ps aux --sort=-%mem | head -10   # top 10 memory consumers

# ─── Target specific process ───────────────────────────────────
ps -p 1234                # info about specific PID
ps -u username            # all processes owned by a user
ps -C nginx               # processes by command name
```

### `ps aux` Output Explained

```
USER       PID  %CPU  %MEM    VSZ    RSS  TTY   STAT  START   TIME  COMMAND
root         1   0.0   0.1  16952   1076  ?     Ss    10:00   0:01  /sbin/init
www-data  1234   2.3   1.4 512340  28432  ?     S     10:05   0:15  nginx: worker
```

| Column | Meaning |
|---|---|
| `USER` | Process owner |
| `PID` | Process ID |
| `%CPU` | CPU usage percentage |
| `%MEM` | Memory usage percentage |
| `VSZ` | Virtual memory size (KB) — total memory mapped |
| `RSS` | Resident Set Size (KB) — actual RAM used right now |
| `TTY` | Terminal (`?` = no terminal, background daemon) |
| `STAT` | Process state (R/S/D/T/Z + modifiers) |
| `TIME` | Total CPU time consumed |
| `COMMAND` | Command that started the process |

> **RSS vs VSZ:** RSS is what actually matters for memory pressure — it's the real RAM in use. VSZ includes memory mapped but not loaded yet. A process with high VSZ but low RSS is fine; high RSS means real memory consumption.

---

## `top` — Real-Time System Monitor

```bash
top               # launch real-time process monitor
top -u username   # show only processes for a specific user
top -p 1234       # monitor specific PID only
top -b -n 1       # batch mode, one snapshot — good for scripts/logging
```

### `top` Interactive Keys (Used During Live Session)

| Key | Action |
|---|---|
| `P` | Sort by CPU usage |
| `M` | Sort by memory usage |
| `T` | Sort by running time |
| `k` | Kill a process (prompts for PID then signal) |
| `r` | Renice a process (change priority) |
| `1` | Toggle per-CPU breakdown |
| `u` | Filter by user |
| `f` | Field management (add/remove columns) |
| `q` | Quit |
| `Space` | Force refresh |

### `top` Header Explained

```
top - 14:22:10 up 5 days,  3:12,  2 users,  load average: 0.52, 0.48, 0.41
Tasks: 183 total,   1 running, 182 sleeping,   0 stopped,   0 zombie
%Cpu(s):  3.2 us,  1.1 sy,  0.0 ni, 95.2 id,  0.3 wa,  0.0 hi,  0.2 si
MiB Mem :  15882.3 total,   4231.1 free,   8012.4 used,   3638.8 buff/cache
MiB Swap:   2048.0 total,   2048.0 free,      0.0 used.   7200.5 avail Mem
```

| Field | Meaning |
|---|---|
| `load average` | CPU queue depth at 1m, 5m, 15m — above number of CPUs = overloaded |
| `us` | User space CPU % |
| `sy` | Kernel/system CPU % |
| `id` | Idle CPU % — low idle = CPU bottleneck |
| `wa` | I/O wait % — high wa = disk/network bottleneck |
| `buff/cache` | Memory used for disk cache — Linux reclaims this freely |

> **Load average rule of thumb:** On a 4-core machine, load average of 4.0 = 100% utilized, above 4.0 = overloaded. Load average above CPU count = processes waiting in queue.

---

## `htop` — Enhanced Interactive Monitor

`htop` is `top` with a better interface — color-coded bars, mouse support, easier to use.

```bash
htop                  # launch htop
htop -u username      # filter by user
htop -p 1234,5678     # monitor specific PIDs
```

### `htop` vs `top`

| Feature | `top` | `htop` |
|---|---|---|
| Pre-installed | ✅ Always | ❌ Needs install |
| Mouse support | ❌ | ✅ |
| Color bars | ❌ | ✅ |
| Horizontal scroll | ❌ | ✅ |
| Tree view | Manual | Built-in F5 |
| Kill without PID typing | ❌ | ✅ Select + F9 |
| Scripts/automation | ✅ (`-b` mode) | ❌ |

> In interviews and production debugging, mention both — `top` for scripts and quick checks, `htop` for interactive investigation.

---

## Signals — The Core Communication Mechanism

Signals are **software interrupts** sent to processes. The kernel delivers them; the process handles, ignores, or is killed by them.

### The Signals You Must Know

| Signal | Number | Name | Can Catch? | Meaning |
|---|---|---|---|---|
| `SIGHUP` | 1 | Hangup | ✅ | Terminal closed / reload config |
| `SIGINT` | 2 | Interrupt | ✅ | Ctrl+C — polite stop |
| `SIGQUIT` | 3 | Quit | ✅ | Ctrl+\ — quit with core dump |
| `SIGKILL` | 9 | Kill | ❌ | Force kill — cannot be caught or ignored |
| `SIGTERM` | 15 | Terminate | ✅ | Polite termination request (default) |
| `SIGSTOP` | 19 | Stop | ❌ | Pause process — cannot be caught |
| `SIGCONT` | 18 | Continue | ✅ | Resume a stopped process |
| `SIGUSR1` | 10 | User 1 | ✅ | App-defined (often: reload, rotate logs) |
| `SIGUSR2` | 12 | User 2 | ✅ | App-defined |

> **SIGTERM vs SIGKILL:** Always try SIGTERM first — gives the process a chance to clean up (close files, finish requests, flush buffers). SIGKILL is the last resort — it's immediate and cannot be caught, risking data corruption or orphaned resources.

> **SIGHUP real-world use:** Nginx, Apache, and most daemons reload their config on SIGHUP without restarting: `kill -HUP $(cat /var/run/nginx.pid)`

---

## `kill`, `killall`, `pkill` — Terminating Processes

### `kill` — By PID

```bash
kill 1234               # send SIGTERM (default) to PID 1234
kill -9 1234            # send SIGKILL to PID 1234
kill -SIGTERM 1234      # explicit signal name (same as default)
kill -SIGKILL 1234      # same as -9
kill -HUP 1234          # send SIGHUP (reload config)
kill -l                 # list all signal names and numbers

# Kill multiple PIDs at once
kill 1234 5678 9012
```

### `killall` — By Process Name

```bash
killall nginx           # SIGTERM all processes named 'nginx'
killall -9 nginx        # SIGKILL all processes named 'nginx'
killall -HUP nginx      # reload config for all nginx processes
killall -u username     # kill all processes owned by user
```

> `killall` matches the **exact process name**. If the name is truncated in `/proc`, it may miss some processes.

### `pkill` — Pattern-Based Kill (Most Flexible)

```bash
pkill nginx             # SIGTERM processes matching pattern 'nginx'
pkill -9 nginx          # SIGKILL matching processes
pkill -u username       # kill all processes by user
pkill -P 1234           # kill all children of PID 1234
pkill -f "python train.py"  # match against FULL command line (-f flag)

# pgrep — find PIDs without killing (pkill's read-only sibling)
pgrep nginx             # print PIDs matching 'nginx'
pgrep -l nginx          # print PID + name
pgrep -a nginx          # print PID + full command
pgrep -u username       # PIDs for a specific user
```

### When to Use Which

| Scenario | Tool |
|---|---|
| You have the PID | `kill -15 1234` |
| Kill by exact process name | `killall nginx` |
| Kill by partial name or command pattern | `pkill -f "train.py"` |
| Find PIDs before killing | `pgrep -l nginx` |
| Kill all children of a process | `pkill -P 1234` |

---

## Background & Foreground Processes

### Running Processes in Background

```bash
# Start a process in background immediately
python train.py &
# Output: [1] 4821   ← job number in brackets, PID after

# Start long-running job silently in background
nohup python train.py > training.log 2>&1 &
# nohup = don't kill when terminal closes
# > training.log = redirect stdout to file
# 2>&1 = redirect stderr to same file as stdout
# & = run in background
```

### Moving Between Foreground and Background

```bash
# Running foreground process? Pause it:
Ctrl+Z          # sends SIGSTOP — suspends the process

# Check your background/suspended jobs:
jobs            # list all jobs
jobs -l         # list with PIDs

# Resume a job:
fg              # bring most recent job to foreground
fg %1           # bring job number 1 to foreground
bg              # resume most recent job in background
bg %2           # resume job number 2 in background

# Kill a specific job:
kill %1         # kill job number 1 (by job spec, not PID)
```

### `nohup` vs `&` — The Difference

| Tool | What it does | Use when |
|---|---|---|
| `&` | Runs in background of current shell | Short tasks in current session |
| `nohup` | Disconnects from terminal's HUP signal | Long-running jobs that must survive logout |
| `nohup ... &` | Both — background AND survives logout | ML training runs, batch jobs |

```bash
# Common pattern for ML training jobs
nohup python train.py --epochs 100 > train.log 2>&1 &
echo "Training started with PID: $!"   # $! = PID of last background process

# Monitor it later
tail -f train.log
```

---

## Job Control — Full Workflow

```bash
# 1. Start a job
python train.py &
# [1] 4821

# 2. Check running jobs
jobs -l
# [1]+  4821 Running    python train.py

# 3. Bring to foreground to interact
fg %1
# python train.py (now in foreground)

# 4. Pause it (Ctrl+Z)
# [1]+  Stopped    python train.py

# 5. Check — now stopped
jobs
# [1]+  Stopped    python train.py

# 6. Resume in background
bg %1
# [1]+ python train.py &

# 7. Let it finish or kill it
kill %1
# [1]+  Terminated    python train.py
```

---

## `nice` and `renice` — Process Priority

The kernel uses **nice values** (-20 to +19) to prioritize CPU scheduling.
- Lower nice value = higher priority (gets more CPU)
- Higher nice value = lower priority (yields CPU to others)
- Default nice value = 0
- Only root can set negative nice values

```bash
# Start a process with lower priority (nice to others)
nice -n 10 python train.py      # nice value +10, lower priority

# Start with higher priority (root only)
sudo nice -n -5 python serve.py # nice value -5, higher priority

# Change priority of a running process
renice 10 -p 1234               # set PID 1234 to nice +10
renice 10 -u username           # lower priority for all user's processes
sudo renice -5 -p 1234          # increase priority (root required)
```

> In MLOps: run training jobs with `nice -n 10` so they don't starve serving processes. Keep inference APIs at default (0) or higher priority.

---

## `/proc` Filesystem — Process Internals

Every process has a directory at `/proc/PID/` exposing its internals.

```bash
ls /proc/1234/
# cmdline  cwd  environ  exe  fd  maps  mem  net  root  stat  status

cat /proc/1234/cmdline    # full command that started the process (null-separated)
cat /proc/1234/status     # human-readable process status (state, memory, PIDs)
cat /proc/1234/environ    # environment variables the process sees
ls -la /proc/1234/fd      # open file descriptors (files, sockets, pipes)
cat /proc/1234/maps       # memory map (what's loaded where in virtual memory)
readlink /proc/1234/exe   # path to the executable binary
readlink /proc/1234/cwd   # current working directory of the process
```

> `/proc/PID/fd/` shows **every open file, socket, and pipe** a process has. Useful for debugging "too many open files" errors and understanding what a process is actually doing.

---

## Common Interview Questions & Answers

**Q: What is the difference between SIGTERM and SIGKILL?**
> SIGTERM (15) is a polite request — the process can catch it, finish in-flight work, close connections, and exit cleanly. SIGKILL (9) is a kernel-level force kill — it cannot be caught, blocked, or ignored. Always use SIGTERM first and give the process time to clean up. SIGKILL risks data corruption, orphaned locks, and incomplete transactions.

**Q: What is a zombie process and how do you remove it?**
> A zombie process has finished executing but its entry remains in the process table because its parent hasn't called `wait()` to read the exit code. You can't kill a zombie directly — it's already dead. You either kill the parent (which triggers cleanup), or if the parent is gone, PID 1 (init/systemd) adopts and reaps it. Persistent zombies indicate a bug in the parent process.

**Q: What is a daemon process?**
> A daemon is a background process not attached to any terminal (TTY shows `?` in `ps aux`). It runs continuously serving requests — nginx, sshd, systemd. Daemons typically start at boot, run as specific users, and handle SIGHUP to reload config without restarting.

**Q: What does `nohup` do and when do you use it?**
> `nohup` makes a process immune to SIGHUP — the signal sent when a terminal closes. Without it, any background job started in an SSH session dies when you log out. Use `nohup ... &` for long-running jobs like ML training that must survive session disconnection.

**Q: What is load average and how do you interpret it?**
> Load average is the average number of processes in the CPU run queue (running + waiting for CPU) over 1, 5, and 15 minutes. Compare it to your CPU count: on a 4-core system, load average of 4.0 means fully utilized, above 4.0 means processes are waiting. High `wa` (I/O wait) in `top` with high load means the bottleneck is disk or network, not CPU.

**Q: What is the difference between `ps aux` and `ps -ef`?**
> Both show all processes but use different syntax conventions. `ps aux` is BSD-style and includes `%CPU`, `%MEM`, `VSZ`, `RSS` columns. `ps -ef` is UNIX-style and includes `PPID` (parent PID) and start time. In practice, `ps aux` is more commonly used for resource inspection; `ps -ef` is used when you need to see process hierarchy via PPID.

---

## MLOps / DevOps Real-World Patterns

```bash
# Find what's eating CPU right now
ps aux --sort=-%cpu | head -10

# Find what's consuming most memory
ps aux --sort=-%mem | head -10

# Kill a stuck ML training job by script name
pkill -f "python train.py"

# Start training, survive SSH disconnect, log output
nohup python train.py --config cfg.yaml > /logs/train.log 2>&1 &
echo "PID: $!"

# Monitor log in real time
tail -f /logs/train.log

# Check if a service is running
pgrep -l nginx || echo "nginx not running"

# Reload nginx config without restart
kill -HUP $(cat /var/run/nginx.pid)
# or
pkill -HUP nginx

# Gracefully stop a service
kill -TERM $(pgrep gunicorn)
sleep 5
pgrep gunicorn && kill -9 $(pgrep gunicorn)   # force if still running

# Find process holding a port
ss -tlnp | grep :8080
# or
lsof -i :8080
```

---

## Side-by-Side Cheat Sheet

| Task | Command |
|---|---|
| Snapshot all processes | `ps aux` |
| Real-time monitor | `top` / `htop` |
| Find process by name | `pgrep -l nginx` |
| Find process by command | `pgrep -af "train.py"` |
| Polite kill by PID | `kill -15 1234` |
| Force kill by PID | `kill -9 1234` |
| Kill by name | `killall nginx` |
| Kill by pattern | `pkill -f "train.py"` |
| Run in background | `command &` |
| Survive logout | `nohup command &` |
| Suspend foreground job | `Ctrl+Z` |
| List jobs | `jobs -l` |
| Resume in background | `bg %1` |
| Bring to foreground | `fg %1` |
| Lower process priority | `nice -n 10 command` |
| Change running priority | `renice 10 -p 1234` |
| Reload config (no restart) | `kill -HUP $(pgrep nginx)` |

---

## One-Line Revision Summary

> Every process has a PID, state, and owner — `ps aux` snapshots them, `top`/`htop` monitors them live — signals are software interrupts where SIGTERM (15) allows graceful shutdown and SIGKILL (9) cannot be caught — `kill` targets by PID, `killall` by name, `pkill` by pattern — `&` backgrounds a job, `Ctrl+Z` suspends it, `fg`/`bg` move it, `nohup` makes it survive logout — `nice`/`renice` control CPU priority.

---

## Interview Delivery Tips

1. **SIGTERM before SIGKILL — always justify it:** "I always try SIGTERM first to give the process time to flush buffers and close connections — SIGKILL risks data corruption."
2. **Zombie explanation is a classic** — the examiner wants to hear "you can't kill a zombie directly, kill the parent" — most candidates just say "kill -9 it" which is wrong.
3. **`pgrep -f` vs `pkill`** — mentioning the `-f` flag (match against full command line) shows you know the difference between matching the process name vs the entire command including arguments.
4. **`nohup` in MLOps context** — "for long training runs I use `nohup python train.py > train.log 2>&1 &` so jobs survive SSH disconnections" — this is immediately relatable.
5. **Load average interpretation** — "compare it to CPU count" is the key insight most people skip.
6. **`/proc/PID/fd`** — mentioning this for debugging "too many open files" errors shows systems-level depth that stands out.
7. **`kill -HUP` for config reload** — "most daemons reload config on SIGHUP without downtime" is a production operations insight, not just a signal number fact.